# 🐱 vs 🐶 — Cats & Dogs CNN Classifier

**Task 2 | Deep Learning Workflow — End to End**

This notebook walks through every step of training a Convolutional Neural Network
to tell cats from dogs:

1. Environment setup & imports  
2. Dataset preparation & exploration  
3. Data augmentation with `ImageDataGenerator`  
4. CNN architecture  
5. Training with callbacks  
6. Validation curves (accuracy & loss)  
7. Confusion matrix  
8. Feature-map visualisation  
9. Single-image prediction  

---
*Dataset: [Kaggle Dogs vs. Cats](https://www.kaggle.com/c/dogs-vs-cats) — or the
sample set generated by `setup_data.py`.*


## 1 — Imports & Configuration

In [ ]:
import os, random, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    GlobalAveragePooling2D, Dense, Dropout,
)
from tensorflow.keras import regularizers
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, load_img, img_to_array,
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger,
)
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Hyper-parameters — tweak these to experiment
IMG_SIZE   = 150      # height = width in pixels
BATCH_SIZE = 32
EPOCHS     = 25       # increase for the full dataset
LR         = 1e-3

DATA_DIR   = "data"   # created by setup_data.py
MODEL_DIR  = "models"
PLOTS_DIR  = "plots"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f"TensorFlow {tf.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")


## 2 — Dataset Preparation

Run `python setup_data.py` once before executing this notebook.  
That script either downloads a small sample set or organises your Kaggle download
into the structure `ImageDataGenerator` expects:

```
data/
    train/
        cats/   *.jpg
        dogs/   *.jpg
    val/
        cats/   *.jpg
        dogs/   *.jpg
```


In [ ]:
# Quick sanity check — how many images do we have?
for split in ("train", "val"):
    for cls in ("cats", "dogs"):
        path = os.path.join(DATA_DIR, split, cls)
        if os.path.isdir(path):
            n = len(os.listdir(path))
            print(f"  {split:5s} / {cls:5s}: {n:>6,} images")
        else:
            print(f"  [missing] {path}")


### 2.1 — Browse a few images

In [ ]:
def show_sample_grid(data_dir, split="train", n_per_class=6):
    fig, axes = plt.subplots(2, n_per_class, figsize=(n_per_class * 2.5, 5.5))
    for row, cls in enumerate(("cats", "dogs")):
        folder = os.path.join(data_dir, split, cls)
        files  = random.sample(os.listdir(folder), min(n_per_class, len(os.listdir(folder))))
        for col, fname in enumerate(files[:n_per_class]):
            img = load_img(os.path.join(folder, fname), target_size=(IMG_SIZE, IMG_SIZE))
            axes[row][col].imshow(img)
            axes[row][col].axis("off")
            if col == 0:
                axes[row][col].set_ylabel(cls.capitalize(), fontsize=12,
                                          fontweight="bold", labelpad=5)
    fig.suptitle(f"Sample Images — {split} set", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

show_sample_grid(DATA_DIR)


## 3 — Data Augmentation

Augmentation artificially expands the training set by randomly transforming
existing images.  It's one of the most effective regularisation tricks for
small image datasets.

| Transform         | Why it helps |
|-------------------|--------------|
| Horizontal flip   | Cats/dogs can face either direction |
| ±15° rotation     | Slightly tilted photos are common |
| ±10 % shift       | Subject isn't always centred |
| 15 % zoom         | Variable distances from camera |
| Shear             | Handles perspective distortion |

The **validation** generator uses *only* rescaling — we want an honest view of
how the model performs on unmodified data.


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest",
)
val_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "train"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    seed=SEED,
)
val_gen = val_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "val"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
    seed=SEED,
)

print("Class index mapping:", train_gen.class_indices)
# {'cats': 0, 'dogs': 1}  →  model output > 0.5 means "dog"


### 3.1 — Visualise augmented images

In [ ]:
# Pull one batch and display the first 8 augmented images.
batch_imgs, batch_labels = next(train_gen)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
class_names = {0: "Cat", 1: "Dog"}
for i, ax in enumerate(axes.flatten()):
    ax.imshow(batch_imgs[i])
    ax.set_title(class_names[int(batch_labels[i])], fontsize=11)
    ax.axis("off")
fig.suptitle("Augmented Training Batch (pixel values rescaled to [0,1])",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 4 — CNN Architecture

The network has **four convolutional blocks**, each followed by Batch
Normalisation and Max Pooling.  After the convolutional stack we use
**Global Average Pooling** instead of Flatten → Dense, which significantly
reduces the parameter count and generalises better on limited data.

```
Input (150 × 150 × 3)
  ↓  Conv Block 1 : Conv2D(32)  + BN + MaxPool → 75 × 75 × 32
  ↓  Conv Block 2 : Conv2D(64)  + BN + MaxPool → 37 × 37 × 64
  ↓  Conv Block 3 : Conv2D(128) + BN + MaxPool → 18 × 18 × 128
  ↓  Conv Block 4 : Conv2D(256) + BN + MaxPool →  9 ×  9 × 256
  ↓  GlobalAvgPool                             →       256
  ↓  Dense(256, relu) + Dropout(0.5)
  ↓  Dense(1, sigmoid)  →  P(dog)
```


In [ ]:
L2 = 1e-4   # weight-decay strength

def conv_block(x, filters):
    x = Conv2D(filters, 3, padding="same", activation="relu",
               kernel_regularizer=regularizers.l2(L2))(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2)(x)
    return x

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = conv_block(inputs, 32)
x = conv_block(x, 64)
x = conv_block(x, 128)
x = conv_block(x, 256)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu", kernel_regularizer=regularizers.l2(L2))(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs, name="CatDog_CNN")
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


## 5 — Training

Three callbacks make the training loop smarter:

* **ModelCheckpoint** — saves weights whenever val_accuracy improves
* **ReduceLROnPlateau** — halves the learning rate after 4 plateau epochs
* **EarlyStopping** — aborts training if val_loss stalls for 8 epochs


In [ ]:
best_path = os.path.join(MODEL_DIR, "best_model.keras")

callbacks = [
    ModelCheckpoint(best_path, monitor="val_accuracy",
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor="val_loss", patience=8,
                  restore_best_weights=True, verbose=1),
    CSVLogger(os.path.join(MODEL_DIR, "training_log.csv")),
]

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1,
)

# Save final weights too (the best checkpoint is already at best_path).
model.save(os.path.join(MODEL_DIR, "final_model.keras"))
print("Training done.")


## 6 — Validation Curves

These two plots are the first thing to look at after training:

* If training accuracy diverges above validation accuracy → **overfitting**
  (try more dropout, more augmentation, or a smaller model).
* If both curves plateau early → **underfitting**
  (try a deeper model or more epochs).


In [ ]:
def plot_training_curves(history, save_dir=None):
    epochs_ran = range(1, len(history.history["accuracy"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    ax1.plot(epochs_ran, history.history["accuracy"],    "b-o",
             ms=5, label="Train")
    ax1.plot(epochs_ran, history.history["val_accuracy"], "r-o",
             ms=5, label="Validation")
    ax1.set_title("Accuracy", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
    ax1.legend(); ax1.grid(alpha=0.3)

    # Loss
    ax2.plot(epochs_ran, history.history["loss"],     "b-o", ms=5, label="Train")
    ax2.plot(epochs_ran, history.history["val_loss"], "r-o", ms=5, label="Validation")
    ax2.set_title("Loss", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Binary Cross-Entropy")
    ax2.legend(); ax2.grid(alpha=0.3)

    fig.suptitle("Training Curves — Cats vs. Dogs CNN",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    if save_dir:
        out = os.path.join(save_dir, "training_curves.png")
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"Saved → {out}")
    plt.show()
    plt.close(fig)

plot_training_curves(history, save_dir=PLOTS_DIR)


## 7 — Evaluation on the Validation Set

In [ ]:
# Reload the best saved weights for evaluation.
model = load_model(best_path)

val_gen.reset()   # rewind to the first batch
y_scores = model.predict(val_gen, verbose=1).ravel()
y_pred   = (y_scores > 0.5).astype(int)
y_true   = val_gen.classes

class_names = list(val_gen.class_indices.keys())   # ['cats', 'dogs']

acc = np.mean(y_pred == y_true)
print(f"Validation accuracy : {acc:.4f}  ({acc*100:.1f} %)")
print(f"Total images        : {len(y_true)}")
print(f"Correct             : {int(np.sum(y_pred == y_true))}")
print(f"Incorrect           : {int(np.sum(y_pred != y_true))}")
print()
print(classification_report(y_true, y_pred, target_names=class_names))


### 7.1 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted label", fontsize=12)
ax.set_ylabel("True label", fontsize=12)
ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
out = os.path.join(PLOTS_DIR, "confusion_matrix.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")


### 7.2 — ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_scores)
auc = roc_auc_score(y_true, y_scores)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, "b-", lw=2, label=f"CNN  (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curve — Cats vs. Dogs", fontsize=14, fontweight="bold")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "roc_curve.png"), dpi=150, bbox_inches="tight")
plt.show()


### 7.3 — Misclassified Examples

In [ ]:
wrong_idx = np.where(y_true != y_pred)[0]
print(f"Total misclassified: {len(wrong_idx)} / {len(y_true)}")

if len(wrong_idx) > 0:
    n_show  = min(12, len(wrong_idx))
    chosen  = np.random.choice(wrong_idx, n_show, replace=False)
    cols, rows = 4, (n_show + 3) // 4

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()

    for ax, idx in zip(axes, chosen):
        img_path = os.path.join(val_gen.directory, val_gen.filenames[idx])
        img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        ax.imshow(img)
        true_lbl = class_names[y_true[idx]]
        pred_lbl = class_names[y_pred[idx]]
        ax.set_title(f"True: {true_lbl}\nPred: {pred_lbl}",
                     fontsize=9, color="red")
        ax.axis("off")
    for ax in axes[n_show:]:
        ax.axis("off")

    fig.suptitle("Misclassified Images", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "misclassified.png"), dpi=150,
                bbox_inches="tight")
    plt.show()


## 8 — Feature-Map Visualisation

This section peers inside the network to show what each filter in the first
convolutional layer responds to.  Early layers tend to detect simple patterns
like edges and colours; deeper layers combine those into complex shapes.


In [ ]:
# Pick any image to feed through.
sample_dir = os.path.join(DATA_DIR, "val", "cats")
sample_imgs = [f for f in os.listdir(sample_dir)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))]

if sample_imgs:
    sample_path = os.path.join(sample_dir, sample_imgs[0])
    orig_img = load_img(sample_path, target_size=(IMG_SIZE, IMG_SIZE))
    x = img_to_array(orig_img) / 255.0
    x = np.expand_dims(x, axis=0)

    # Build an intermediate model that outputs the first Conv2D activations.
    first_conv = next(l for l in model.layers if "conv2d" in l.name)
    feat_model = Model(inputs=model.input, outputs=first_conv.output)
    features   = feat_model.predict(x, verbose=0)   # (1, H, W, filters)

    # Display the original and the first 32 feature maps.
    n_maps = min(32, features.shape[-1])
    cols, rows = 8, n_maps // 8

    fig = plt.figure(figsize=(18, rows * 2.2 + 2.5))
    # Original image at the top.
    ax0 = fig.add_subplot(1, 1, 1)
    plt.tight_layout()

    fig, axes = plt.subplots(rows + 1, cols, figsize=(cols * 2, (rows + 1) * 2))
    # First row: original image centred.
    for j in range(cols):
        axes[0][j].axis("off")
    axes[0][cols // 2 - 1].imshow(orig_img)
    axes[0][cols // 2 - 1].set_title("Input image", fontsize=9)

    for i in range(n_maps):
        r, c = (i // cols) + 1, i % cols
        axes[r][c].imshow(features[0, :, :, i], cmap="viridis")
        axes[r][c].set_title(f"filter {i}", fontsize=7)
        axes[r][c].axis("off")

    fig.suptitle(f"First Conv Layer ({first_conv.name}) — Feature Maps",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "feature_maps.png"), dpi=150,
                bbox_inches="tight")
    plt.show()
else:
    print("No sample images found — run setup_data.py first.")


## 9 — Single-Image Prediction

Drop any `.jpg` path into `IMAGE_PATH` below to see what the model thinks.


In [ ]:
def predict_image(model, img_path, img_size=IMG_SIZE):
    \"\"\"
    Predict cat or dog for a single image and display the result.
    \"\"\"
    img = load_img(img_path, target_size=(img_size, img_size))
    x   = img_to_array(img) / 255.0
    x   = np.expand_dims(x, axis=0)
    score = float(model.predict(x, verbose=0)[0][0])

    if score > 0.5:
        label, conf = "Dog", score
        color = "#3498db"
    else:
        label, conf = "Cat", 1.0 - score
        color = "#e74c3c"

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img)
    ax.set_title(f"{label}  ({conf*100:.1f} % confident)",
                 fontsize=14, fontweight="bold", color=color)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    return label, conf

# ── Change this path to any image you like ──────────────────────────
sample_dir  = os.path.join(DATA_DIR, "val", "cats")
sample_imgs = [f for f in os.listdir(sample_dir)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))]

if sample_imgs:
    IMAGE_PATH = os.path.join(sample_dir, sample_imgs[0])
    label, conf = predict_image(model, IMAGE_PATH)
    print(f"Prediction: {label}  (confidence: {conf*100:.1f} %)")
else:
    print("Add an image path to IMAGE_PATH and rerun.")


## 10 — Summary & Next Steps

**What we covered in this notebook**

| Step | Concept |
|------|---------|
| `ImageDataGenerator` | Data loading, rescaling, augmentation |
| Conv2D + MaxPooling2D | Spatial feature extraction |
| BatchNormalization + Dropout | Regularisation |
| ModelCheckpoint + EarlyStopping + ReduceLROnPlateau | Smart training callbacks |
| Accuracy / loss curves | Diagnosing overfitting |
| Confusion matrix + ROC | Detailed performance breakdown |
| Feature-map visualisation | Network interpretability |

**Ideas to try next**

* **Transfer learning** — swap the Conv stack for a pretrained MobileNetV2
  or EfficientNet backbone (`tf.keras.applications`) and fine-tune only the
  head.  You'll likely jump 5–10 % accuracy with the same data.
* **Grad-CAM** — generate heat-maps that highlight *which part* of the image
  the model looked at when making its prediction.
* **Larger input** — increase `IMG_SIZE` to 224 × 224 and add one more Conv
  block.
* **Full Kaggle dataset** — run `setup_data.py --source kaggle` with all 25 000
  images for a meaningful accuracy benchmark.
